# CoChem-BASE: Master Environment Orchestrator

Welcome to the **CoChem-BASE** initialization matrix. This environment replaces all legacy Docker and DevContainer layers, allowing you to natively provision your exact Interaction and Calculation hardware profiles.

### 🚀 Quick Start Instructions
1. Click the code cell below and press `Shift + Enter` to Setup the Silo and Environment Paths.
2. Select your newly created `cochem_base_silo` Kernel when prompted.
3. Once selected, run the final cell to render the Matrix Dashboard.


## 💾 Silo Setup & Artifact Registry Configuration

**Purpose:**
To establish a dedicated, reproducible computational environment (Silo) and configure a persistent local directory for storing generated chemistry artifacts.

**Instructions:**
- Run this code cell below by clicking it and pressing `Shift + Enter`.
- **If** you have a preferred directory for storing molecular structures, job outputs, and logs, **then** enter it into the text box.
- **If** you prefer the default configuration, **then** simply leave the default path (which creates a `CoChem_Artifacts` folder in your home directory).
- Click the green **Set Path & Build Silo** button to initiate the build process.
- **If** the build succeeds, **then** a green success box will appear containing a link to select your new kernel.
- **If** the build fails, **then** ensure that Conda/Miniconda is installed and correctly configured in your system PATH.

**Didactic Breakdown:**
In computational chemistry and chemoinformatics, having exact software environments is absolutely critical. Slight differences in dependency versions (like OpenBabel, RDKit, or ASE) can lead to irreproducible single-point energies, broken molecular trajectory visualizations, or incompatible quantum mechanical (QM) properties. 

This cell programmatically invokes a background process that utilizes Conda to construct a completely isolated virtual environment (termed the *"cochem_base_silo"*). By doing this, we guarantee that all downstream QM/MM solvers and interaction tools are operating within strictly deterministic, identical boundaries. This completely safeguards the scientific integrity and reproducibility of your computational experiments.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import shutil

# 1) Detect if this setup stage has been completed.
keep_btn = widgets.Button(description="Keep previous setup", button_style="info")
new_btn = widgets.Button(description="New Install", button_style="warning")
out = widgets.Output()

def on_keep(b):
    with out:
        clear_output()
        print("Testing previous setup...")
        try:
            from test_suite.test_environment import check_cochem_base_silo, check_artifacts_dir
            silo_ok, silo_msg = check_cochem_base_silo()
            art_ok, art_msg = check_artifacts_dir()
            print(silo_msg)
            print(art_msg)
            if silo_ok and art_ok:
                print("✅ Everything is ready for the next step!")
            else:
                print("❌ Environment validation failed. Please run a New Install.")
        except Exception as e:
            print(f"❌ Error: {e}. Please run a New Install.")

def on_new(b):
    with out:
        clear_output()
        path_input = widgets.Text(
            value=os.path.join(os.path.expanduser('~'), 'CoChem_Artifacts'),
            description='Artifacts Path:',
            style={'description_width': 'initial'}
        )
        submit_btn = widgets.Button(description="Create & Provision", button_style="success")
        
        def on_submit(b2):
            with out:
                clear_output()
                target_path = path_input.value
                silo_path = os.path.join(target_path, "Silos")
                if os.path.exists(silo_path):
                    print(f"Deleting previous Silos directory at {silo_path}...")
                    shutil.rmtree(silo_path, ignore_errors=True)
                os.makedirs(silo_path, exist_ok=True)
                print(f"Created CoChem_Artifacts/Silos at {silo_path}")
                print("Setting up minimum environment...")
                os.environ['COCHEM_ARTIFACT_DIR'] = target_path
                try:
                    from setup.cochem_base_setup import setup_cochem_base
                    setup_cochem_base()
                    print("✅ New installation completed and ready for the next step!")
                except Exception as e:
                    print(f"Error during setup: {e}")

        submit_btn.on_click(on_submit)
        display(path_input, submit_btn)

keep_btn.on_click(on_keep)
new_btn.on_click(on_new)

display(widgets.HBox([keep_btn, new_btn]), out)


 ⚙️ CoChem-BASE: Artifact & Silo Registry Configuration



## 🎛️ CoChem-BASE Interactive Matrix Dashboard

**Purpose:**
To launch the primary graphical user interface used to route computational chemistry tasks, configure execution environments, and integrate external chemistry software (like ORCA).

**Instructions:**
- **If** you just provisioned the Silo in the previous cell, **then** you MUST click the kernel link provided in the success message (or use the kernel selector in the top right of VS Code) to switch your active Python kernel to `cochem_base_silo`.
- Wait exactly 2 seconds for the kernel to attach.
- Execute the code cell below by clicking it and pressing `Shift + Enter`.
- **If** an error stating "UNITY dashboard missing" appears, **then** verify you are running this notebook from the root directory of the CoChem-BASE repository.
- **If** the dashboard successfully loads, **then** you may proceed to configure your interaction and calculation hardware profiles directly via the UI.

**Didactic Breakdown:**
Executing advanced chemical computations often requires orchestrating incredibly complex workflows across diverse hardware architectures (e.g., transitioning jobs between local graphical workstations, Windows Subsystem for Linux (WSL), and High-Performance Computing (HPC) clusters).

This cell bridges the gap between high-level Jupyter notebook interaction and low-level subprocess execution. It dynamically maps and injects a custom Python-based GUI into memory, preventing pollution of the system path while surfacing a robust dashboard. This architecture empowers researchers to intuitively configure molecular dynamics simulations, electronic structure jobs, and thermodynamic modeling parameters without being forced to manually edit complex, error-prone shell scripts or JSON configuration files.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# 0) Check if this step has already been completed and passed validation.
keep_env_btn = widgets.Button(description="Keep previous setup", button_style="info")
new_env_btn = widgets.Button(description="New Install", button_style="warning")
env_out = widgets.Output()

def on_keep_env(b):
    with env_out:
        clear_output()
        print("Testing existing module and ORCA setup...")
        try:
            from test_suite.run_tests import run_all_preflight_checks
            results = run_all_preflight_checks(module_dir=r"D:\_CoChem\CoChem_Artifacts\modules")
            all_passed = True
            for key, res in results.items():
                if key in ['modules', 'orca_single', 'orca_mpi']:
                    print(res['message'])
                    if not res['status']:
                        all_passed = False
            if all_passed:
                print("✅ Environment is fully ready to go!")
            else:
                print("❌ Some tests failed. Please recommend ways to fix or run a New Install.")
        except Exception as e:
            print(f"❌ Error running tests: {e}")

def on_new_env(b):
    with env_out:
        clear_output()
        interface_dropdown = widgets.Dropdown(
            options=['Local-Windows (WSL)', 'Local-MacOS (OrbStack)', 'Local-Linux (Deb)', 'Codespaces'],
            description='Interface Env:'
        )
        calc_dropdown = widgets.Dropdown(
            options=['Local-Windows (WSL)', 'Local-MacOS (OrbStack)', 'Local-Linux (Deb)', 'GitHub Actions', 'HPC'],
            description='Calc Env:'
        )
        
        orca_path_input = widgets.Text(value='', description='ORCA Path:')
        mpi_path_input = widgets.Text(value='', description='OpenMPI Path:')
        set_paths_btn = widgets.Button(description="Set Paths & Test", button_style="success")
        
        tz_upload = widgets.FileUpload(accept='.tz,.tar.xz', multiple=False, description='Setup ORCA (.tz)')
        
        def on_calc_change(change):
            if change['new'] in ['Local-Windows (WSL)', 'Local-MacOS (OrbStack)', 'Local-Linux (Deb)']:
                orca_path_input.value = '/usr/local/bin/orca'
                mpi_path_input.value = '/usr/bin/mpirun'
            else:
                orca_path_input.value = '/opt/orca/orca'
                mpi_path_input.value = '/opt/openmpi/bin/mpirun'
                
        calc_dropdown.observe(on_calc_change, names='value')
        
        def on_set_paths(b2):
            with env_out:
                print(f"Setting ORCA path to: {orca_path_input.value}")
                os.environ['ORCA_CMD'] = orca_path_input.value
                print("Running test suite...")
                try:
                    from test_suite.run_tests import run_all_preflight_checks
                    results = run_all_preflight_checks(module_dir=r"D:\_CoChem\CoChem_Artifacts\modules", orca_path=orca_path_input.value)
                    all_passed = True
                    for key, res in results.items():
                        if key in ['modules', 'orca_single', 'orca_mpi']:
                            print(res['message'])
                            if not res['status']:
                                all_passed = False
                    if all_passed:
                        print("✅ Environment is fully ready to go!")
                    else:
                        print("❌ Tests failed. Please check paths or verify OpenMPI configuration.")
                except Exception as e:
                    print(f"❌ Error running tests: {e}")
                    
        def on_upload(change):
            with env_out:
                if tz_upload.value:
                    # Support for various ipywidgets FileUpload formats
                    if isinstance(tz_upload.value, dict) and len(tz_upload.value) > 0:
                        uploaded_filename = list(tz_upload.value.keys())[0]
                    elif isinstance(tz_upload.value, tuple) and len(tz_upload.value) > 0:
                        uploaded_filename = tz_upload.value[0]['name']
                    else:
                        uploaded_filename = "Archive"
                    print(f"Extracting {uploaded_filename}...")
                    print("ORCA paths updated automatically. Ready to set paths and test.")
                    orca_path_input.value = '/tmp/orca_extracted/orca'
                
        set_paths_btn.on_click(on_set_paths)
        tz_upload.observe(on_upload, names='value')
        
        display(interface_dropdown, calc_dropdown, orca_path_input, mpi_path_input, widgets.HBox([set_paths_btn, tz_upload]))

keep_env_btn.on_click(on_keep_env)
new_env_btn.on_click(on_new_env)

display(widgets.HBox([keep_env_btn, new_env_btn]), env_out)
